# BS5: fresh rerun with corrected export
Upload the latest source ZIP as ONE Kaggle Dataset, attach only that source Dataset, enable GPU, then Run All.
New C-R/C-N x seeds551/552, 60k/run (240k total), followed by frozen-best eval600 (cap25200).
The corrected readers are checked BEFORE training. Output uses a separate fresh folder; old results are preserved.
Download the final ZIP. For a saved Dataset, retain its identical `.zip.bin` copy and transport JSON to protect container bytes.
No training is executed while building this notebook.


In [ ]:
import os, sys, json, pathlib, subprocess, hashlib
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
# Walk Dataset aliases with cycle protection; avoid training/result trees.
found, seen = {}, set()
for base, directories, filenames in os.walk(INPUT, followlinks=True):
    folder = pathlib.Path(base)
    real = folder.resolve()
    if real in seen:
        directories[:] = []
        continue
    seen.add(real)
    if folder.name == 'evaluation' and folder.parent.name == 'scripts' and 'run_bs5_retrain_eval600.py' in filenames:
        source = folder.parents[1]
        found.setdefault(source.resolve(), source)
    directories[:] = [name for name in directories if name not in
                      ('continuous600', 'trajectories', 'control_raw', 'runs', 'parents',
                       'results', 'checkpoints', 'archive', '.git', '.codegraph', '__pycache__')
                      and not name.startswith('repro_')]
sources = sorted(found.values())
assert len(sources) == 1, f'Attach ONE latest source Dataset only; found: {sources}'
SRC = sources[0]
os.chdir(SRC)
OUT = WORK / 'bs5_retrain_eval600_fresh'
def command(*args):
    subprocess.run([sys.executable, '-B', '-u', '-m', 'scripts.evaluation.run_bs5_retrain_eval600',
                    *map(str, args)], cwd=SRC, check=True)
print('Source:', SRC, 'Output:', OUT)
print('NEW training240k; separate eval cap25200; no historical checkpoints required.')


In [ ]:
# Refuse an old source Dataset before spending a training budget.
EXPECTED_READERS = {'scripts/analysis/analyze_bs5_continuous.py': '700a3e23f1f647ced1d02501c60e2a5526a5fa5d0ebd0edc904ec354ff2965d3', 'scripts/analysis/analyze_bs5_curriculum_extend.py': '5afbc87ff7da20a1391216f64046b508a58cf3807427d53b10c5883d4d551701'}
for relative, expected in EXPECTED_READERS.items():
    path = SRC / relative
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, (
        f'Wrong source version: {relative}. Attach the latest source ZIP supplied with this fresh notebook.'
    )
print('Corrected export readers verified.')
import torch, numpy as np
print('Runtime:', sys.version, torch.__version__, np.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings.'
if not (OUT / 'workflow.json').is_file():
    assert not OUT.exists(), 'Retain partial preparation; use a clean session rather than overwrite evidence.'
    command('prepare', '--output', OUT, '--device', 'cuda')
else:
    print('This fresh lineage already exists: continue its committed work, without preparing another budget.')


In [ ]:
# Existing training runner: NEW prefix24k -> OWN tail36k, then best validation150.
# Best weights are frozen before continuous600. No training updates during evaluation.
# Re-running skips completed work. An interrupted eval game cannot be reset/retried.
command('run','--output',OUT)


In [ ]:
from IPython.display import FileLink, display
import shutil
receipt = OUT / 'delivery_receipt.json'
assert receipt.is_file(), 'Not complete. Preserve working output and inspect the preceding error.'
info = json.loads(receipt.read_text())
package = pathlib.Path(info['path'])
assert info['accepted'] and info['includes_selected_weights']
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
assert sha256_file(package) == info['sha256']
# Same ZIP bytes under a non-archive extension for preservation in a saved Dataset.
protected = package.with_suffix('.zip.bin')
if not protected.exists():
    try:
        os.link(package, protected)
    except OSError:
        shutil.copyfile(package, protected)
assert sha256_file(protected) == info['sha256']
transport = package.with_name(package.stem + '_transport.json')
transport.write_text(json.dumps(dict(
    original_filename=package.name, protected_filename=protected.name,
    sha256=info['sha256'], bytes=package.stat().st_size,
    restore='Copy the .zip.bin file to the original .zip filename; preserve bytes, do not repack.',
), indent=2), encoding='utf-8')
os.chdir(WORK)
print('DONE: final ZIP contains training audit + repro with four selected best weights + full eval600 evidence.')
print('Download the ZIP now. If saving output as a Dataset, also retain .zip.bin and transport JSON.')
for path in (package, protected, transport, receipt):
    print(path.name, f'{path.stat().st_size / 1024**2:.2f} MiB')
    display(FileLink(path.relative_to(WORK).as_posix()))
